In [1]:
import requests
import time
import pandas as pd
import os
import json
import ast

In [2]:
# For PGS training inforamtion

def get_method(score: json) -> str:
    temp = score.get("method_name", "None")
    return str(temp)

def get_variants_number(score: json) -> int:
    temp = score.get("variants_number", 0)
    return temp

def get_date_release(score: json) -> str:
    temp = score.get("date_release", "None")
    return str(temp)

def get_sample_number(score: json) -> int:

    if len(score.get("samples_variants", [])) > 0:
        temp = score.get("samples_variants", [])
        return temp[0].get("sample_number", 0)
    
    elif len(score.get("samples_training", [])) > 0:
        temp = score.get("samples_training", [])
        return temp[0].get("sample_number", 0)
    
    return 0

def get_sample_case(score: json) -> int:

    if len(score.get("samples_variants", [])) > 0:
        temp = score.get("samples_variants", [])
        return temp[0].get("sample_number", 0)
    
    elif len(score.get("samples_training", [])) > 0:
        temp = score.get("samples_training", [])
        return temp[0].get("sample_number", 0)
    
    return 0

def get_sample_controls(score: json) -> int:

    if len(score.get("samples_variants", [])) > 0:
        temp = score.get("samples_variants", [])
        return temp[0].get("sample_controls", 0)
    
    elif len(score.get("samples_training", [])) > 0:
        temp = score.get("samples_training", [])
        return temp[0].get("sample_controls", 0)
    
    return 0

def get_ancestry_broad(score: json) -> str:

    if len(score.get("samples_variants", [])) > 0:
        temp = score.get("samples_variants", [])
        return temp[0].get("ancestry_broad", "None")
    
    elif len(score.get("samples_training", [])) > 0:
        temp = score.get("samples_training", [])
        return temp[0].get("ancestry_broad", "None")
    
    return "None"

def get_cohort(score: json) -> str:

    if len(score.get("samples_variants", [])) > 0:
        temp = score.get("samples_variants", [])
        if len(temp[0].get("cohorts", [])) > 0:
            temp = temp[0].get("cohorts", [])
            return temp[0].get("name_full", "None")

    elif len(score.get("samples_training", [])) > 0:
        temp = score.get("samples_training", [])
        if len(temp[0].get("cohorts", [])) > 0:
            temp = temp[0].get("cohorts", [])
            return temp[0].get("name_full", "None")
    
    return "None"

In [3]:
# For PGS evaluation inforamtion 
def get_eval_num(score: json) -> int:
      temp = score.get("count", 0)
      return int(temp)

def get_eval_sample(score: json) -> list:
      result = []
      temp = score.get("results", [])
      if len(temp) == 0:
            return result
      for t in temp:
            temp2 = t.get("sampleset").get("samples")[0]
            result.append(temp2.get("sample_number"))
      return result

def get_eval_control(score: json) -> list:
      result = []
      temp = score.get("results", [])
      if len(temp) == 0:
            return result
      for t in temp:
            temp2 = t.get("sampleset").get("samples")[0]
            result.append(temp2.get("sample_controls"))
      return result

def get_eval_case(score: json) -> list:
      result = []
      temp = score.get("results", [])
      if len(temp) == 0:
            return result
      for t in temp:
            temp2 = t.get("sampleset").get("samples")[0]
            result.append(temp2.get("sample_cases"))
      return result

def get_eval_ancestry(score: json) -> list:
      result = []
      temp = score.get("results", [])
      if len(temp) == 0:
            return result
      for t in temp:
            temp2 = t.get("sampleset").get("samples")[0]
            result.append(temp2.get("ancestry_broad"))
      return result

def get_eval_cohort(score: json) -> list:
      result = []
      temp = score.get("results", [])
      if len(temp) == 0:
            return result
      for t in temp:
            temp2 = t.get("sampleset").get("samples")[0]
            temp3 = temp2.get("cohorts", [])
            if len(temp3) != 0:
                  result.append(temp3[0].get("name_full"))
                  continue
            result.append("None")
      return result

def get_eval_metrics(score: json) -> list:
      result = []
      temp = score.get("results", [])
      if len(temp) == 0:
            return result
      for t in temp:
            temp2 = t.get("performance_metrics").get("effect_sizes", [])
            if len(temp2) != 0:
                  result.append(temp2[0].get("name_long"))
                  continue
            temp2 = t.get("performance_metrics").get("class_acc", [])
            if len(temp2) != 0:
                  result.append(temp2[0].get("name_long"))
                  continue
            temp2 = t.get("performance_metrics").get("othermetrics", [])
            if len(temp2) != 0:
                  result.append(temp2[0].get("name_long"))
                  continue
            result.append("None")
      return result

def get_eval_covariates(score: json) -> list:
      result = []
      temp = score.get("results", [])
      if len(temp) == 0:
            return result
      for t in temp:
            result.append(t.get("covariates"))
      return result

def get_eval_estimate(score: json) -> list:
      result = []
      temp = score.get("results", [])
      if len(temp) == 0:
            return result
      for t in temp:
            temp2 = t.get("performance_metrics").get("effect_sizes", [])
            if len(temp2) != 0:
                  result.append(temp2[0].get("estimate", 0))
                  continue
            temp2 = t.get("performance_metrics").get("class_acc", [])
            if len(temp2) != 0:
                  result.append(temp2[0].get("estimate", 0))
                  continue
            temp2 = t.get("performance_metrics").get("othermetrics", [])
            if len(temp2) != 0:
                  result.append(temp2[0].get("estimate", 0))
                  continue
            result.append(0)
      return result


In [7]:
# Read contained icd and description
pgs_id_list = pd.read_csv(os.path.join(os.getcwd(), "pgs_id_list_260225.csv"), index_col=None)

#records = []
#processed_pgs = []
for idx, row in pgs_id_list.iterrows():
    pgs_ids_str = row["pgs_ids"]
    pgs_ids = ast.literal_eval(pgs_ids_str)

    for pgs_id in pgs_ids:
        if pgs_id in processed_pgs:
            continue
        temp = "https://www.pgscatalog.org/rest/score/" + pgs_id
        r = requests.get(temp)
        data = r.json()

        temp = "https://www.pgscatalog.org/rest/performance/search?pgs_id=" + pgs_id
        r = requests.get(temp)
        perform_data = r.json()

        records.append({
            "PGS_ID": pgs_id,
            "num_variant": get_variants_number(data),
            "training_ancestry": get_ancestry_broad(data),
            "training_method": get_method(data),
            "training_cohort": get_cohort(data),
            "num_training_sample": get_sample_number(data),
            "num_training_controls": get_sample_controls(data),
            "num_training_cases": get_sample_case(data),
            "date_release": get_date_release(data),
            "num_eval": get_eval_num(perform_data),
            "num_eval_sample": get_eval_sample(perform_data),
            "num_eval_controls": get_eval_control(perform_data),
            "num_eval_cases": get_eval_case(perform_data),
            "eval_ancestry": get_eval_ancestry(perform_data),
            "eval_cohort": get_eval_cohort(perform_data),
            "eval_metrics": get_eval_metrics(perform_data),
            "eval_covariates": get_eval_covariates(perform_data),
            "eval_estimate": get_eval_estimate(perform_data)
        })
        processed_pgs.append(pgs_id)
        time.sleep(0.2)
    print(f"Finish processing {row['loinc']}")

df = pd.DataFrame(records)
output_path = "pgs_metadata_260225.csv"
df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

Finish processing 1869-7
Finish processing 1869-7
Finish processing 1884-6
Finish processing 704-7
Finish processing 706-2
Finish processing 39156-5
Finish processing 29463-7
Finish processing 3141-9
Finish processing 30522-7
Finish processing 8462-4
Finish processing 711-2
Finish processing 713-8
Finish processing 789-8
Finish processing 2243-4
Finish processing 3184-9
Finish processing 4548-4
Finish processing 4544-3
Finish processing 718-7
Finish processing 2085-9
Finish processing 10835-7
Finish processing 18262-6
Finish processing 26474-7
Finish processing 26478-8
Finish processing 28539-5
Finish processing 786-4
Finish processing 742-7
Finish processing 26485-3
Finish processing 751-8
Finish processing 26511-6
Finish processing 47228-2
Finish processing 32207-3
Finish processing 71693-6
Finish processing 777-3
Finish processing 35177-5
Finish processing 788-0
Finish processing 8480-6
Finish processing 2986-8
Finish processing 9830-1
Finish processing 2571-8
Finish processing 3084

In [6]:
print(processed_pgs)

['PGS001888', 'PGS002101', 'PGS000671', 'PGS003491', 'PGS000672', 'PGS001889', 'PGS002102', 'PGS003492', 'PGS000164', 'PGS004730', 'PGS004729', 'PGS005175', 'PGS005181', 'PGS001377', 'PGS003945', 'PGS004727', 'PGS000163', 'PGS003940', 'PGS001378', 'PGS000088', 'PGS004728', 'PGS000089', 'PGS004220', 'PGS005114', 'PGS004733', 'PGS002840', 'PGS003887', 'PGS002630', 'PGS004982', 'PGS003462', 'PGS003130', 'PGS002851', 'PGS004900', 'PGS000298', 'PGS004096', 'PGS004319', 'PGS005199', 'PGS002856', 'PGS005200', 'PGS000716', 'PGS004378', 'PGS002751', 'PGS004080', 'PGS004050', 'PGS002313', 'PGS004992', 'PGS005279', 'PGS002843', 'PGS002853', 'PGS005204', 'PGS004989', 'PGS003897', 'PGS002841', 'PGS003843', 'PGS002855', 'PGS004218', 'PGS001825', 'PGS002846', 'PGS000770', 'PGS000829', 'PGS001943', 'PGS004409', 'PGS004985', 'PGS003126', 'PGS004216', 'PGS004987', 'PGS005202', 'PGS003884', 'PGS004022', 'PGS005198', 'PGS002532', 'PGS002857', 'PGS002434', 'PGS004901', 'PGS004736']


In [12]:
temp = "https://www.pgscatalog.org/rest/performance/search?pgs_id=PGS003972"
r = requests.get(temp)
score = r.json()
print(json.dumps(score, indent=2, ensure_ascii=False))

{
  "size": 3,
  "count": 3,
  "next": null,
  "previous": null,
  "results": [
    {
      "id": "PPM019134",
      "associated_pgs_id": "PGS003972",
      "phenotyping_reported": "Abdominal aortic aneurysm",
      "publication": {
        "id": "PGP000513",
        "title": "Genome-wide association meta-analysis identifies risk loci for abdominal aortic aneurysm and highlights PCSK9 as a therapeutic target.",
        "doi": "10.1038/s41588-023-01510-y",
        "PMID": 37845353,
        "journal": "Nat Genet",
        "firstauthor": "Roychowdhury T",
        "date_publication": "2023-10-16"
      },
      "sampleset": {
        "id": "PSS011199",
        "samples": [
          {
            "sample_number": 6940,
            "sample_cases": 1130,
            "sample_controls": 5810,
            "sample_percent_male": null,
            "sample_age": null,
            "phenotyping_free": null,
            "followup_time": null,
            "ancestry_broad": "European",
            "anc